In [1]:
if (!require("microbenchmark")) install.packages("microbenchmark", repos = "https://cloud.r-project.org")
library(microbenchmark)


Loading required package: microbenchmark

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘microbenchmark’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 1. Load the UCI Heart Disease dataset


In [2]:
url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

col_names <- c("age","sex","cp","trestbps","chol","fbs","restecg","thalach",
               "exang","oldpeak","slope","ca","thal","target")

heart <- tryCatch(
  read.csv(url, header = FALSE, col.names = col_names, na.strings = "?"),
  error = function(e) {
    message("Could not download from UCI (no internet in this environment?). Generating a synthetic fallback dataset instead.")
    set.seed(42)
    n <- 303
    data.frame(
      age = sample(29:77, n, replace = TRUE),
      sex = sample(0:1, n, replace = TRUE),
      cp = sample(0:3, n, replace = TRUE),
      trestbps = round(rnorm(n, 130, 17)),
      chol = round(rnorm(n, 246, 51)),
      fbs = sample(0:1, n, replace = TRUE),
      restecg = sample(0:2, n, replace = TRUE),
      thalach = round(rnorm(n, 150, 23)),
      exang = sample(0:1, n, replace = TRUE),
      oldpeak = round(runif(n, 0, 4), 1),
      slope = sample(0:2, n, replace = TRUE),
      ca = sample(0:3, n, replace = TRUE),
      thal = sample(c(3,6,7), n, replace = TRUE),
      target = sample(0:1, n, replace = TRUE)
    )
  }
)

str(heart)
head(heart)


'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ target  : int  0 2 1 0 0 0 3 0 2 1 ...


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


## Simulate realistic data-entry problems



In [3]:
set.seed(123)
heart_dirty <- heart
n <- nrow(heart_dirty)

neg_idx     <- sample(1:n, 5)
na_idx      <- sample(setdiff(1:n, neg_idx), 5)
extreme_idx <- sample(setdiff(1:n, c(neg_idx, na_idx)), 5)

heart_dirty$trestbps[neg_idx]     <- -abs(heart_dirty$trestbps[neg_idx])
heart_dirty$trestbps[na_idx]      <- NA
heart_dirty$trestbps[extreme_idx] <- sample(305:400, 5, replace = TRUE)

cat("Injected negative BP at rows:", neg_idx, "\n")
cat("Injected NA BP at rows:", na_idx, "\n")
cat("Injected extreme BP (>300) at rows:", extreme_idx, "\n")

summary(heart_dirty$trestbps)


Injected negative BP at rows: 179 14 195 118 299 
Injected NA BP at rows: 233 248 15 155 91 
Injected extreme BP (>300) at rows: 94 265 204 303 141 


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
 -138.0   120.0   130.0   131.3   140.0   397.0       5 

## 2. Task 1 - BP-cleaning function using if-else



In [4]:
clean_bp_value <- function(bp) {
  if (is.na(bp)) {
    return(NA_real_)
  } else if (bp < 0) {
    return(NA_real_)
  } else if (bp > 250) {
    return(250)
  } else {
    return(bp)
  }
}

heart_dirty$trestbps_cleaned <- sapply(heart_dirty$trestbps, clean_bp_value)

heart_dirty[c(neg_idx, na_idx, extreme_idx), c("trestbps", "trestbps_cleaned")]


,trestbps,trestbps_cleaned
,<dbl>,<dbl>
179,-130,NA
14,-120,NA
195,-120,NA
118,-138,NA
299,-110,NA
233,NA,NA
248,NA,NA
15,NA,NA
155,NA,NA


## 3. Task 2 - Error handling with `tryCatch()`

### 3a. Safely calculate mean BP when missing values are present


In [5]:
safe_mean_bp <- function(x) {
  tryCatch({
    m <- mean(x, na.rm = TRUE)
    if (is.nan(m)) stop("All values are NA -- cannot compute a mean.")
    m
  },
  warning = function(w) {
    message("Warning while computing mean BP: ", conditionMessage(w))
    NA
  },
  error = function(e) {
    message("Error while computing mean BP: ", conditionMessage(e))
    NA
  })
}

cat("Mean of raw (dirty) trestbps: ", safe_mean_bp(heart_dirty$trestbps), "\n")
cat("Mean of cleaned trestbps:     ", safe_mean_bp(heart_dirty$trestbps_cleaned), "\n")

cat("Mean of an all-NA vector:     ", safe_mean_bp(c(NA, NA, NA)), "\n")


Mean of raw (dirty) trestbps:  131.2819 
Mean of cleaned trestbps:      133.8908 


Error while computing mean BP: All values are NA -- cannot compute a mean.



Mean of an all-NA vector:      NA 


### 3b. Safely calculate a ratio `chol / trestbps`, handling zero / NA / invalid denominators

In [6]:
safe_ratio <- function(numerator, denominator) {
  mapply(function(num, den) {
    tryCatch({
      if (is.na(num) || is.na(den)) {
        stop("Numerator or denominator is NA.")
      }
      if (den == 0) {
        stop("Denominator is zero.")
      }
      if (!is.numeric(num) || !is.numeric(den)) {
        stop("Non-numeric input.")
      }
      num / den
    },
    error = function(e) {
      message(sprintf("Skipping ratio (chol=%s, trestbps=%s): %s",
                       as.character(num), as.character(den), conditionMessage(e)))
      NA_real_
    })
  }, numerator, denominator)
}

heart_dirty$chol_bp_ratio <- safe_ratio(heart_dirty$chol, heart_dirty$trestbps_cleaned)

demo_result <- safe_ratio(c(200, 180, NA), c(0, 120, 80))
print(demo_result)

head(heart_dirty[, c("chol", "trestbps_cleaned", "chol_bp_ratio")])


Skipping ratio (chol=263, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=199, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=302, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=183, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=246, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=315, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=211, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=149, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=275, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=264, trestbps=NA): Numerator or denominator is NA.

Skipping ratio (chol=200, trestbps=0): Denominator is zero.

Skipping ratio (chol=NA, trestbps=80): Numerator or denominator is NA.



[1]  NA 1.5  NA


,chol,trestbps_cleaned,chol_bp_ratio
,<dbl>,<dbl>,<dbl>
1,233,145,1.606897
2,286,160,1.787500
3,229,120,1.908333
4,250,130,1.923077
5,204,130,1.569231
6,236,120,1.966667


## 4. Task 3 - Loop-based vs vectorized outlier/invalid-value detection

In [7]:
bp_vec <- heart_dirty$trestbps

loop_invalid <- function(x) {
  flags <- logical(length(x))
  for (i in seq_along(x)) {
    val <- x[i]
    if (is.na(val)) {
      flags[i] <- NA
    } else if (val < 0 || val > 250) {
      flags[i] <- TRUE
    } else {
      flags[i] <- FALSE
    }
  }
  flags
}

vectorized_invalid <- function(x) {
  ifelse(is.na(x), NA, (x < 0 | x > 250))
}

loop_result       <- loop_invalid(bp_vec)
vectorized_result <- vectorized_invalid(bp_vec)

identical(loop_result, vectorized_result)


[1] TRUE

In [8]:
loop_time <- system.time(replicate(200, loop_invalid(bp_vec)))
vec_time  <- system.time(replicate(200, vectorized_invalid(bp_vec)))

cat("Loop-based system.time() (200 reps):\n"); print(loop_time)
cat("\nVectorized system.time() (200 reps):\n"); print(vec_time)

bench <- microbenchmark(
  loop_based = loop_invalid(bp_vec),
  vectorized = vectorized_invalid(bp_vec),
  times = 100
)
print(bench)


Loop-based system.time() (200 reps):
   user  system elapsed 
  0.013   0.000   0.014 

Vectorized system.time() (200 reps):
   user  system elapsed 
  0.007   0.000   0.008 
Unit: microseconds
       expr   min     lq     mean median     uq    max neval
 loop_based 51.28 52.945 60.08109 55.485 60.740 108.33   100
 vectorized 12.25 13.345 15.02018 14.010 15.095  61.82   100


## 5. Task 4 - Validate the cleaned data

In [9]:
cleaned_bp <- heart_dirty$trestbps_cleaned

n_missing <- sum(is.na(cleaned_bp))
bp_min    <- min(cleaned_bp, na.rm = TRUE)
bp_max    <- max(cleaned_bp, na.rm = TRUE)
bp_mean   <- mean(cleaned_bp, na.rm = TRUE)
bp_median <- median(cleaned_bp, na.rm = TRUE)

cat("Missing BP values (NA count):", n_missing, "\n")
cat("Min BP:   ", bp_min, "\n")
cat("Max BP:   ", bp_max, "\n")
cat("Mean BP:  ", round(bp_mean, 2), "\n")
cat("Median BP:", bp_median, "\n\n")

no_negative <- all(cleaned_bp[!is.na(cleaned_bp)] >= 0)
no_over_250 <- all(cleaned_bp[!is.na(cleaned_bp)] <= 250)

cat("No negative values remain: ", no_negative, "\n")
cat("No values above 250 remain:", no_over_250, "\n")

stopifnot(no_negative, no_over_250)
cat("\nValidation PASSED: cleaned trestbps has no negative or >250 values.\n")


Missing BP values (NA count): 10 
Min BP:    94 
Max BP:    250 
Mean BP:   133.89 
Median BP: 130 

No negative values remain:  TRUE 
No values above 250 remain: TRUE 

Validation PASSED: cleaned trestbps has no negative or >250 values.


## 6. Save the cleaned dataset

In [10]:
heart_out <- heart_dirty
heart_out$trestbps <- heart_out$trestbps_cleaned
heart_out$trestbps_cleaned <- NULL

write.csv(heart_out, "cleaned_heart_data.csv", row.names = FALSE)
cat("Saved cleaned_heart_data.csv with", nrow(heart_out), "rows and", ncol(heart_out), "columns.\n")


Saved cleaned_heart_data.csv with 303 rows and 15 columns.


## 7. Conclusion

- The **vectorized** approach for detecting invalid BP values is consistently faster than the **for-loop** approach, because R's vectorized operations (`ifelse`, comparison operators) are implemented in optimized C code and avoid the interpreter overhead of looping element-by-element in R.
- The performance gap grows with dataset size - for small datasets like this one the difference is modest, but on large clinical datasets (tens of thousands of rows) the vectorized method would be dramatically faster and is the recommended approach for production data-cleaning pipelines.
- `tryCatch()` allowed the pipeline to gracefully handle all-NA vectors and zero/NA denominators without crashing, instead emitting informative messages - this is essential for robust, production-grade healthcare data pipelines where a single bad row should not halt the entire analysis.

**Efficiency verdict: Vectorized operations are more efficient than explicit for-loops for this data-cleaning task, both in code brevity and execution speed.**
